# OCTRON YOLO prediction - the "simple" python workflow

This notebook shows the simplest python-first way to run OCTRON prediction and tracking without opening the GUI.

It uses `octron.tools.predict.run_predict()`, which is the same high-level wrapper used by the command line (`octron predict`). Use this notebook when you want a compact, scriptable workflow with sensible defaults.

For lower-level control (custom progress handling, custom tracker parameters, and custom region measurement functions), see `OCTRON_YOLO_predict_advanced.ipynb`.

Results loading is intentionally not covered here; see `OCTRON_results_loading.ipynb` for that workflow.


## 1. Imports

`run_predict()` accepts files, folders, or lists of videos.<br> It resolves `device="auto"` to CUDA → MPS → CPU.

In [ ]:
from pathlib import Path

from octron.tools.predict import run_predict
from octron.yolo_octron.constants import DEFAULT_REGION_PROPERTIES, ALL_REGION_PROPERTIES


## 2. Configure paths

Set the videos and trained model. `model_path` may be a direct `.pt` file or a training/run directory containing `best.pt` in a standard OCTRON/YOLO location.


In [ ]:
# One video, a list of videos, or a directory containing .mp4 files.
videos = [
    Path("/Users/horst/Downloads/tomopteris-planktonis-3/test/example_video.mp4"),
]
# videos = Path("path/to/folder_with_mp4s")

# A trained model file, or a directory that contains weights/best.pt.
model_path = Path("/Users/horst/Downloads/tomopteris-planktonis-3/model")

# Final output location. None writes octron_predictions/ next to each video.
output_dir = None
# output_dir = Path("path/to/output_root")

# Optional fast local staging directory. Useful when output_dir is on a network/external drive.
# If None, OCTRON may still use prediction_cache_dir from config.yaml; otherwise caching is off.
local_cache_dir = None
# local_cache_dir = Path("/fast/local/cache")


## 3. Choose device, tracker, and output detail

`device="auto"` is usually the best default. It chooses CUDA when available, then Apple Silicon MPS, then CPU.

`tracker_name` can be any available tracker key/name (for example `bytetrack`, `botsort`, `hybridsort`). For a custom tracker YAML, see the next section.


In [ ]:
device = "auto"  # or: "cuda", "mps", "cpu"
tracker_name = "bytetrack"

# Region measurements to add to the CSV output.
# None = bbox-only (fastest)
region_properties = None

# Standard minimal detailed output (currently area only):
# region_properties = DEFAULT_REGION_PROPERTIES # See ALL_REGION_PROPERTIES to list all available

# Explicit selection (Python equivalent of: octron predict --detailed area,eccentricity,solidity)
# region_properties = ("area", "eccentricity", "solidity")

# All available scalar region properties (Python equivalent of: --detailed all)
# region_properties = tuple(name for group in ALL_REGION_PROPERTIES.values() for name in group)


## 4. Run prediction

This mirrors the CLI command:

```bash
octron predict VIDEO --model MODEL --tracker bytetrack --device auto
```

The wrapper prints progress and returns when all videos are processed.


In [ ]:
run_predict(
    videos=videos,
    model_path=model_path,
    tracker_name=tracker_name,
    device=device,
    conf_thresh=0.5,
    iou_thresh=0.7,
    skip_frames=0,
    one_object_per_label=False,
    opening_radius=0,
    overwrite=True,
    buffer_size=500,
    region_properties=region_properties,
    output_dir=output_dir,
    local_cache_dir=local_cache_dir,
)


## 5. Optional: use a custom tracker config

A good workflow is to dump a default tracker config from the command line, edit it, then pass it here.

```bash
octron dump-tracker-config bytetrack -o my_tracker.yaml
```

When `tracker_cfg_path` is supplied, it overrides `tracker_name`.


In [ ]:
# tracker_cfg_path = Path("my_tracker.yaml")

# run_predict(
#     videos=videos,
#     model_path=model_path,
#     tracker_name=None,
#     tracker_cfg_path=tracker_cfg_path,
#     device=device,
#     overwrite=False,
#     output_dir=output_dir,
#     local_cache_dir=local_cache_dir,
# )


## 6. Next steps

- For advanced use (custom progress handling, `tracker_params`, `extra_properties`), see `OCTRON_YOLO_predict_advanced.ipynb`.
- For loading and analysing prediction output, see `OCTRON_results_loading.ipynb`.
